# The QR algorithm on a $2\times 2$ positive definite matrix

Start with a real symmetric positive definite matrix $A = A^{(0)}$ and iterate
$$
A^{(n)} = Q_nR_n, \qquad A^{(n+1)} = R_nQ_n = Q_n^{\top}A^{(n)}Q_n .
$$
We normalize the factorization so that $R_n$ has positive diagonal entries. Then $Q_n$ is unique, and since
$\det A^{(n)} > 0$ and $\det R_n > 0$, each $Q_n$ is a **rotation**.

Each $A^{(n)}$ is symmetric and positive definite, so the level set
$$
E_n = \{u \in \mathbb{R}^2 : u^{\top}A^{(n)}u = 1\}
$$
is an ellipse. Since $A^{(n+1)} = Q_n^{\top}A^{(n)}Q_n$, we have $E_{n+1} = Q_n^{\top}E_n$: each step just **rotates** the ellipse.
Writing $\bar Q_n = Q_0Q_1\cdots Q_{n-1}$, we get $E_n = \bar Q_n^{\top}E_0$.

If the eigenvalues are $\lambda_1 > \lambda_2 > 0$, then $A^{(n)} \to \operatorname{diag}(\lambda_1,\lambda_2)$, so the ellipse settles into
standard position: semi-axis $1/\sqrt{\lambda_1}$ (the short one) along $e_1$ and $1/\sqrt{\lambda_2}$ (the long one) along $e_2$.
The off-diagonal entry, and the rotation angle at each step, shrink like $(\lambda_2/\lambda_1)^n$.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

plt.rcParams["animation.embed_limit"] = 50  # MB

## Choose the starting matrix

We build $A$ from its eigenvalues and the angle of its first eigenvector, so the convergence rate $\lambda_2/\lambda_1$ is easy to control.
Try making the eigenvalues closer together (slower convergence) or farther apart (faster).

In [2]:
lam1, lam2 = 3.0, 1.8          # eigenvalues, lam1 > lam2 > 0
theta = np.deg2rad(55)          # angle of the lam1-eigenvector

def rotation(phi):
    return np.array([[np.cos(phi), -np.sin(phi)],
                     [np.sin(phi),  np.cos(phi)]])

V = rotation(theta)
A = V @ np.diag([lam1, lam2]) @ V.T
print("A =\n", A.round(4))
print("ratio lam2/lam1 =", lam2 / lam1)

A =
 [[2.1948 0.5638]
 [0.5638 2.6052]]
ratio lam2/lam1 = 0.6


## Run the iteration

In [3]:
def qr_positive(M):
    # QR factorization normalized so that R has positive diagonal
    Q, R = np.linalg.qr(M)
    s = np.sign(np.diag(R))
    s[s == 0] = 1
    return Q * s, s[:, None] * R

n_steps = 20
As = [A]
Qs = []
for n in range(n_steps):
    Q, R = qr_positive(As[-1])
    Qs.append(Q)
    As.append(R @ Q)

# rotation angle of each Q_n
angles = np.array([np.arctan2(Q[1, 0], Q[0, 0]) for Q in Qs])

for n in [0, 1, 2, 5, 10, n_steps]:
    print(f"A^({n}) =\n{As[n].round(6)}\n")

A^(0) =
[[2.194788 0.563816]
 [0.563816 2.605212]]

A^(1) =
[[2.491938 0.592914]
 [0.592914 2.308062]]

A^(2) =
[[2.749117 0.487973]
 [0.487973 2.050883]]

A^(5) =
[[2.985381 0.13164 ]
 [0.13164  1.814619]]

A^(10) =
[[2.999911 0.010362]
 [0.010362 1.800089]]

A^(20) =
[[3.0e+00 6.3e-05]
 [6.3e-05 1.8e+00]]



## Animate the level sets

Between $A^{(n)}$ and $A^{(n+1)}$ we interpolate by rotating through part of the angle $\varphi_n$ of $Q_n$:
$A(t) = R(t\varphi_n)^{\top}A^{(n)}R(t\varphi_n)$ for $0 \le t \le 1$. So the motion you see is exactly the rotation the algorithm performs.

- **Left:** the ellipse $E_n$ (solid), the starting ellipse $E_0$ (dashed), and the principal axes of $E_n$.
- **Right:** $|a^{(n)}_{12}|$ on a log scale, compared with a line of slope $\log(\lambda_2/\lambda_1)$.

In [4]:
#| label: qr-animation
def ellipse(S, m=300):
    # points u with u^T S u = 1, plus the principal semi-axes
    w, U = np.linalg.eigh(S)
    t = np.linspace(0, 2 * np.pi, m)
    pts = U @ (np.vstack([np.cos(t), np.sin(t)]) / np.sqrt(w)[:, None])
    axes = U / np.sqrt(w)  # columns are semi-axis vectors
    return pts, axes

frames_per_step = 6
hold_frames = 15
schedule = [(n, k / frames_per_step) for n in range(n_steps) for k in range(frames_per_step)]
schedule += [(n_steps, 0.0)] * hold_frames

def matrix_at(n, t):
    if n == n_steps:
        return As[n]
    Rt = rotation(t * angles[n])
    return Rt.T @ As[n] @ Rt

off_diag = np.abs([M[0, 1] for M in As])
r = 1.3 / np.sqrt(lam2)

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(8, 4), dpi=72)

pts0, _ = ellipse(A)
ax.plot(*pts0, "--", color="gray", lw=1, label=r"$E_0$")
ax.axhline(0, color="k", lw=0.5)
ax.axvline(0, color="k", lw=0.5)
(curve,) = ax.plot([], [], color="C0", lw=2, label=r"$E_n$")
(ax_short,) = ax.plot([], [], color="C3", lw=2, label=r"axis $1/\sqrt{\lambda_1}$")
(ax_long,) = ax.plot([], [], color="C2", lw=2, label=r"axis $1/\sqrt{\lambda_2}$")
ax.set_xlim(-r, r)
ax.set_ylim(-r, r)
ax.set_aspect("equal")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.06), ncol=2, fontsize=8)
title = ax.set_title("")
mat_text = ax.text(0.03, 0.03, "", transform=ax.transAxes, family="monospace", fontsize=8)

ns = np.arange(n_steps + 1)
ax2.semilogy(ns, off_diag, "o-", color="C0", ms=4, label=r"$|a^{(n)}_{12}|$")
ax2.semilogy(ns, off_diag[-1] * (lam2 / lam1) ** (ns - n_steps), "--", color="gray",
             label=r"slope $\log(\lambda_2/\lambda_1)$")
(marker,) = ax2.plot([], [], "o", color="C3", ms=10)
ax2.set_xlabel("n")
ax2.set_title("Off-diagonal entry")
ax2.legend(fontsize=8)
ax2.grid(True, which="both", alpha=0.3)
fig.tight_layout()
fig.subplots_adjust(bottom=0.22)

def update(i):
    n, t = schedule[i]
    S = matrix_at(n, t)
    pts, axes = ellipse(S)
    curve.set_data(*pts)
    # eigh sorts eigenvalues ascending: column 0 is lam2 (long axis), column 1 is lam1 (short axis)
    ax_long.set_data([-axes[0, 0], axes[0, 0]], [-axes[1, 0], axes[1, 0]])
    ax_short.set_data([-axes[0, 1], axes[0, 1]], [-axes[1, 1], axes[1, 1]])
    title.set_text(f"QR step n = {n}" + (f" → {n + 1}" if 0 < t else ""))
    mat_text.set_text(f"A =\n[{S[0,0]:8.4f} {S[0,1]:8.4f}]\n[{S[1,0]:8.4f} {S[1,1]:8.4f}]")
    marker.set_data([n], [off_diag[n]])
    return curve, ax_short, ax_long, title, mat_text, marker

anim = animation.FuncAnimation(fig, update, frames=len(schedule), interval=60, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

## What to notice

- The ellipse never changes shape: every $A^{(n)}$ has the same eigenvalues, so the semi-axes always have lengths $1/\sqrt{\lambda_1}$ and $1/\sqrt{\lambda_2}$. The algorithm only rotates it.
- The short axis (the $\lambda_1$ direction) swings onto the $e_1$-axis. This is the power method: the first column of $\bar Q_n$ is the
  normalized vector $A^n e_1$, which turns toward the dominant eigenvector, and $E_n = \bar Q_n^{\top}E_0$ carries that eigenvector to $e_1$.
- The rotation angles shrink geometrically, so the motion slows down. On the right, $|a^{(n)}_{12}|$ falls on a line of slope
  $\log(\lambda_2/\lambda_1)$.
- If $\lambda_1 = \lambda_2$ the ellipse is a circle, $A$ is already a multiple of $I$, and nothing moves.

In [5]:
# The accumulated rotation Q_bar_n converges to the eigenvector matrix (up to sign)
Qbar = np.eye(2)
for Q in Qs:
    Qbar = Qbar @ Q
print("Q_bar =\n", Qbar.round(6))
print("eigenvectors V =\n", V.round(6))

Q_bar =
 [[ 0.573619 -0.819122]
 [ 0.819122  0.573619]]
eigenvectors V =
 [[ 0.573576 -0.819152]
 [ 0.819152  0.573576]]
